# Ebird — Bird Observations & Species

Demonstrates **get_recent_observations**, **get_taxonomy**, and **get_species_list** tools.

---
## Setup

Make sure you have installed the dev dependencies and demo requirements:
```bash
pip install -e ".[dev]"
pip install -r demos/requirements.txt
```

Then copy and edit the environment file:
```bash
cp demos/.env.example demos/.env
# edit demos/.env with your X_EBIRD_API_TOKEN
```

In [ ]:
!pip install -q -e "../.."
!pip install -q -r ../requirements.txt

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from demos.shared.llm_setup import get_llm
from pymcpx.ebird import EbirdToolkit

llm = get_llm()

tools = EbirdToolkit().get_tools()

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a birding assistant. Use the ebird tools to answer questions "
            "about bird observations and species.\n\n"
            "Available tools:\n"
            "- get_recent_observations(region_code, max_results) — recent bird sightings\n"
            "- get_taxonomy(search, limit) — search bird species\n"
            "- get_species_list(region_code, search, limit) — species in a region",
        ),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
print("\u2705 Agent ready. Running scenarios...")

In [ ]:
result = executor.invoke({"input": "Search for 'finch' in eBird taxonomy."})
print(result["output"])

In [ ]:
result = executor.invoke(
    {"input": "What birds have been spotted in New York (US-NY) recently? Show me up to 5."}
)
print(result["output"])